# 📊 Caso Banco Berka — Pipeline de Preparación de Datos v3

---

## 🎯 Caso de Uso

> **"Predecir qué clientes tienen alta probabilidad de ser buenos candidatos para una tarjeta de crédito."**

---

## 📋 Tablas utilizadas en esta versión (v3)

| Tabla | Archivo | Obligatoria | Rol en el análisis |
|-------|---------|:-----------:|--------------------|
| `client` | `client.asc` | ✅ | Datos demográficos del cliente (edad, género) |
| `disposition` | `disp.asc` | ✅ | Relación cliente-cuenta; filtramos solo `OWNER` |
| `account` | `account.asc` | ✅ | Características de la cuenta (antigüedad, frecuencia) |
| `transaction` | `trans.asc` | ✅ | Comportamiento financiero (>1M registros agregados) |
| `card` | `card.asc` | ✅ | **Variable objetivo** → `has_card` (0/1) |
| `loan` | `loan.asc` | ⚙️ Opcional | Historial crediticio del cliente |
| `district` | `district.asc` | ⚙️ Opcional | Contexto socioeconómico del distrito del cliente |

> **Diferencia v2 → v3:** Se reincorpora `district.asc` como tabla **opcional**, aportando variables del contexto
> socioeconómico del distrito (salario promedio, desempleo, criminalidad, región). Se mantiene el feature
> `deposit_ratio` introducido en v2. El pipeline maneja los valores implícitos `?` de Jesenik automáticamente.

---

## 🗺️ Mapa del Notebook — Metodología CRISP-DM

```
┌──────────────────────────────────────────────────────────────┐
│  SECCIÓN 0 │ Configuración del entorno                       │
├──────────────────────────────────────────────────────────────┤
│  SECCIÓN 1 │ CRISP-DM: Comprensión de Datos                 │
│            │ → Carga y exploración de 7 tablas               │
├──────────────────────────────────────────────────────────────┤
│  SECCIÓN 2 │ CRISP-DM: Preparación — Limpieza por tabla     │
│            │ → client, account, district (nulos implícitos ?)│
├──────────────────────────────────────────────────────────────┤
│  SECCIÓN 3 │ CRISP-DM: Preparación — Feature Engineering    │
│            │ → Agregación de trans y loan                    │
├──────────────────────────────────────────────────────────────┤
│  SECCIÓN 4 │ CRISP-DM: Preparación — Integración            │
│            │ → Merge de 5+2 tablas, creación has_card        │
├──────────────────────────────────────────────────────────────┤
│  SECCIÓN 5 │ CRISP-DM: Preparación — para Modelado          │
│            │ → Nulos → Outliers → Z-Score → One-Hot          │
├──────────────────────────────────────────────────────────────┤
│  SECCIÓN 6 │ CRISP-DM: Pre-Modelado                         │
│            │ → Exportar tabla minable CSV                    │
└──────────────────────────────────────────────────────────────┘
```

---
## 🔧 SECCIÓN 0 — Configuración del Entorno
**Fase CRISP-DM: Preparación técnica (transversal a todo el proceso)**

Definimos las rutas de trabajo, cargamos las librerías y activamos los flags de tablas opcionales.

> 💡 **Google Colab:** Sube tu carpeta `data/` comprimida en `.zip` y descomenta `!unzip`.

In [1]:
# ==============================================================================
# SECCIÓN 0 — CONFIGURACIÓN DEL ENTORNO
# ==============================================================================
# Fase CRISP-DM: Preparación técnica
#
# Para Google Colab: sube el archivo data.zip y descomenta:
# !unzip data.zip -d ./data

import os
import pandas as pd
import numpy as np

# ── Rutas ──────────────────────────────────────────────────────────────────────
# LOCAL (Anaconda / JupyterLab)
DATA_DIR = r"Berka Bank dataset"

# COLAB (descomenta si corres en Google Colab)
# DATA_DIR = "data"

OUTPUT_CSV = "minable_banco_berka_v3.csv"

# ── Flags de tablas opcionales ─────────────────────────────────────────────────
# Cambia a False si quieres excluir alguna tabla opcional del análisis
USE_LOAN     = True   # Incluir historial de préstamos
USE_DISTRICT = True   # Incluir contexto socioeconómico del distrito

print(f"✅ Entorno configurado correctamente.")
print(f"   📂 Datos en     : {DATA_DIR}")
print(f"   💾 Salida en    : {OUTPUT_CSV}")
print(f"   ⚙️  Loan activo : {USE_LOAN}")
print(f"   ⚙️  District act: {USE_DISTRICT}")

✅ Entorno configurado correctamente.
   📂 Datos en     : Berka Bank dataset
   💾 Salida en    : minable_banco_berka_v3.csv
   ⚙️  Loan activo : True
   ⚙️  District act: True


---
## 📂 SECCIÓN 1 — Carga y Comprensión de los Datos
**Fase CRISP-DM: 🔍 Comprensión de los Datos (Data Understanding)**

Cargamos las 7 tablas y las exploramos para entender su estructura y relaciones.

### Diagrama de relaciones entre tablas (v3):
```
 client ──┐
          ├── disp (OWNER) ──── account ──── trans     ← comportamiento financiero
          │      │           │           └── loan      ← historial crediticio (opcional)
          │      │           └── card                 ← VARIABLE OBJETIVO (has_card)
          └── district_id → district                  ← contexto socioeconómico (opcional)
```

### Aporte de cada tabla al caso de uso:
| Tabla | Variables clave que aporta | Tipo |
|-------|---------------------------|------|
| `client` | `gender`, `age` | Demográfica |
| `disp` | Filtro `OWNER` | Relacional |
| `account` | `frequency`, `account_age_months` | Cuenta |
| `trans` | `avg_balance`, `net_flow`, `deposit_ratio` | Comportamiento financiero |
| `card` | `has_card` (objetivo), `card_type` | **Variable objetivo** |
| `loan` *(opc.)* | `loan_status`, `loan_amount` | Historial crediticio |
| `district` *(opc.)* | `average_salary`, `unemployment_rate`, `region` | Socioeconómica |

In [2]:
# ==============================================================================
# SECCIÓN 1.1 — CARGA DE TABLAS
# ==============================================================================
# Fase CRISP-DM: Comprensión de los Datos
#
# Los archivos .asc son CSV con separador ';'.
# Cargamos las 7 tablas. Las opcionales se cargan según los flags.

def load_data(data_dir, use_loan=True, use_district=True):
    """Carga las tablas del Banco Berka. Opcionales controladas por flags."""
    print("📥 Cargando tablas del Banco Berka...")

    # Tablas obligatorias
    client  = pd.read_csv(os.path.join(data_dir, "client.asc"),  sep=";", low_memory=False)
    disp    = pd.read_csv(os.path.join(data_dir, "disp.asc"),    sep=";", low_memory=False)
    account = pd.read_csv(os.path.join(data_dir, "account.asc"), sep=";", low_memory=False)
    trans   = pd.read_csv(os.path.join(data_dir, "trans.asc"),   sep=";", low_memory=False)
    card    = pd.read_csv(os.path.join(data_dir, "card.asc"),    sep=";", low_memory=False)

    print(f"   ✅ client   : {len(client):>7,} filas")
    print(f"   ✅ disp     : {len(disp):>7,} filas")
    print(f"   ✅ account  : {len(account):>7,} filas")
    print(f"   ✅ trans    : {len(trans):>7,} filas")
    print(f"   ✅ card     : {len(card):>7,} filas")

    # Tabla opcional: loan
    loan = None
    if use_loan:
        loan = pd.read_csv(os.path.join(data_dir, "loan.asc"), sep=";", low_memory=False)
        print(f"   ⚙️  loan     : {len(loan):>7,} filas  (OPCIONAL — activa)")
    else:
        print(f"   ⏭️  loan     : omitida  (flag USE_LOAN=False)")

    # Tabla opcional: district
    district = None
    if use_district:
        district = pd.read_csv(os.path.join(data_dir, "district.asc"), sep=";", low_memory=False)
        print(f"   ⚙️  district : {len(district):>7,} filas  (OPCIONAL — activa)")
    else:
        print(f"   ⏭️  district : omitida  (flag USE_DISTRICT=False)")

    return client, disp, account, trans, card, loan, district


client, disp, account, trans, card, loan, district = load_data(
    DATA_DIR, use_loan=USE_LOAN, use_district=USE_DISTRICT
)

📥 Cargando tablas del Banco Berka...
   ✅ client   :   5,369 filas
   ✅ disp     :   5,369 filas
   ✅ account  :   4,500 filas
   ✅ trans    : 1,056,320 filas
   ✅ card     :     892 filas
   ⚙️  loan     :     682 filas  (OPCIONAL — activa)
   ⚙️  district :      77 filas  (OPCIONAL — activa)


In [3]:
# ==============================================================================
# SECCIÓN 1.2 — EXPLORACIÓN INICIAL
# ==============================================================================
# Fase CRISP-DM: Comprensión de los Datos — ¿Qué contiene cada tabla?

print("=" * 60)
print("EXPLORACIÓN: client — primeras filas")
print("=" * 60)
print(client.head(3).to_string())

print("\n" + "=" * 60)
print("EXPLORACIÓN: disp — tipos de disposición")
print("=" * 60)
print(disp['type'].value_counts())

print("\n" + "=" * 60)
print("EXPLORACIÓN: card — tipos de tarjeta")
print("=" * 60)
print(card['type'].value_counts())

if district is not None:
    print("\n" + "=" * 60)
    print("EXPLORACIÓN: district — calidad de datos (buscar '?')")
    print("=" * 60)
    print(f"   Columnas: {list(district.columns)}")
    print(f"   Filas con '?' en A12: {(district['A12'] == '?').sum()}")
    print(f"   Filas con '?' en A15: {(district['A15'] == '?').sum()}")
    print("   → Estos valores serán imputados con la mediana en la Sección 2.")

EXPLORACIÓN: client — primeras filas
   client_id  birth_number  district_id
0          1        706213           18
1          2        450204            1
2          3        406009            1

EXPLORACIÓN: disp — tipos de disposición
type
OWNER        4500
DISPONENT     869
Name: count, dtype: int64

EXPLORACIÓN: card — tipos de tarjeta
type
classic    659
junior     145
gold        88
Name: count, dtype: int64

EXPLORACIÓN: district — calidad de datos (buscar '?')
   Columnas: ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16']
   Filas con '?' en A12: 1
   Filas con '?' en A15: 1
   → Estos valores serán imputados con la mediana en la Sección 2.


In [4]:
# ==============================================================================
# SECCIÓN 1.3 — ANÁLISIS DE LA VARIABLE OBJETIVO
# ==============================================================================
# Fase CRISP-DM: Comprensión de los Datos — Balance de clases en has_card

print("=" * 60)
print("VARIABLE OBJETIVO: has_card (¿tiene tarjeta de crédito?)")
print("=" * 60)

owners_preview = disp[disp['type'] == 'OWNER'].copy()
card_preview   = owners_preview.merge(card, on='disp_id', how='left')

con_tarjeta = card_preview['card_id'].notna().sum()
sin_tarjeta = card_preview['card_id'].isna().sum()
total       = con_tarjeta + sin_tarjeta

print(f"  ✅ has_card = 1 (con tarjeta) : {con_tarjeta:,}  ({con_tarjeta/total*100:.1f}%)")
print(f"  ❌ has_card = 0 (sin tarjeta) : {sin_tarjeta:,} ({sin_tarjeta/total*100:.1f}%)")
print(f"  Total clientes OWNER          : {total:,}")
print()
ratio = sin_tarjeta / con_tarjeta
print(f"  Ratio desbalance              : {ratio:.1f}:1  (sin/con tarjeta)")
if ratio > 3:
    print("  ⚠️  Desbalance de clases detectado (ratio > 3:1).")
    print("      Considerar SMOTE, class_weight='balanced' o ajuste de umbral.")

VARIABLE OBJETIVO: has_card (¿tiene tarjeta de crédito?)
  ✅ has_card = 1 (con tarjeta) : 892  (19.8%)
  ❌ has_card = 0 (sin tarjeta) : 3,608 (80.2%)
  Total clientes OWNER          : 4,500

  Ratio desbalance              : 4.0:1  (sin/con tarjeta)
  ⚠️  Desbalance de clases detectado (ratio > 3:1).
      Considerar SMOTE, class_weight='balanced' o ajuste de umbral.


---
## 🧹 SECCIÓN 2 — Limpieza y Transformación por Tabla
**Fase CRISP-DM: 🛠️ Preparación de los Datos — Limpieza**

Aplicamos transformaciones específicas sobre cada tabla. En v3 se agrega la limpieza de `district`:

| Tabla | Problema detectado | Solución |
|-------|-------------------|---------|
| `client` | `birth_number` codifica género + mes + año | Decodificar → `gender`, `age` |
| `account` | `frequency` en checo; `date` como entero YYMMDD | Traducir; calcular `account_age_months` |
| `district` *(v3)* | Valores `?` en A12 y A15 (nulos implícitos del distrito Jesenik) | Reemplazar `?` → `NaN` → imputar con mediana |

In [5]:
# ==============================================================================
# SECCIÓN 2.1 — LIMPIEZA DE CLIENTES
# ==============================================================================
# Fase CRISP-DM: Preparación de Datos
#
# El campo 'birth_number' tiene formato YYMMDD.
# Para mujeres, el mes está incrementado en 50 (ej: enero=51).
# Extraemos: género (M/F), mes, año y edad al corte de 1998.
#
# RELEVANCIA: 'gender' y 'age' son predictores del perfil de propensión
# a adoptar una tarjeta de crédito.

def clean_client(df_client):
    """Decodifica birth_number → gender, birth_month, birth_year, age."""
    df = df_client.copy()

    bn = df['birth_number'].astype(str).str.zfill(6)
    yy = bn.str[:2].astype(int)
    mm = bn.str[2:4].astype(int)

    df['gender']      = np.where(mm > 50, 'F', 'M')
    df['birth_month'] = np.where(mm > 50, mm - 50, mm)
    df['birth_year']  = 1900 + yy
    df['age']         = 1998 - df['birth_year']

    df = df.drop(columns=['birth_number'])

    print(f"✅ clean_client: {len(df):,} clientes")
    print(f"   Género       : {df['gender'].value_counts().to_dict()}")
    print(f"   Edad (min/max/media): {df['age'].min()} / {df['age'].max()} / {df['age'].mean():.1f}")
    return df


client_clean = clean_client(client)

✅ clean_client: 5,369 clientes
   Género       : {'M': 2724, 'F': 2645}
   Edad (min/max/media): 11 / 87 / 44.8


In [6]:
# ==============================================================================
# SECCIÓN 2.2 — LIMPIEZA DE CUENTAS
# ==============================================================================
# Fase CRISP-DM: Preparación de Datos
#
# Traduce 'frequency' del checo al español.
# Calcula 'account_age_months' a partir de la fecha de apertura.
#
# RELEVANCIA: La frecuencia del extracto indica el nivel de control financiero.
# La antigüedad de la cuenta refleja la estabilidad del cliente con el banco.

def clean_account(df_account):
    """Traduce frequency y calcula antigüedad de la cuenta en meses."""
    df = df_account.copy()

    freq_map = {
        'POPLATEK MESICNE'  : 'Mensual',
        'POPLATEK TYDNE'    : 'Semanal',
        'POPLATEK PO OBRATU': 'PorTransaccion'
    }
    df['frequency'] = df['frequency'].map(freq_map).fillna('Mensual')

    acc_year  = 1900 + (df['date'] // 10000)
    acc_month = (df['date'] // 100) % 100
    df['account_age_months'] = (1998 - acc_year) * 12 + (12 - acc_month)

    df = df.drop(columns=['date'])

    print(f"✅ clean_account: {len(df):,} cuentas")
    print(f"   Frecuencia: {df['frequency'].value_counts().to_dict()}")
    print(f"   Antigüedad (min/max/media meses): "
          f"{df['account_age_months'].min()} / {df['account_age_months'].max()} / "
          f"{df['account_age_months'].mean():.1f}")
    return df


account_clean = clean_account(account)

✅ clean_account: 4,500 cuentas
   Frecuencia: {'Mensual': 4167, 'Semanal': 240, 'PorTransaccion': 93}
   Antigüedad (min/max/media meses): 12 / 71 / 40.2


In [7]:
# ==============================================================================
# SECCIÓN 2.3 — LIMPIEZA DE DISTRITOS (OPCIONAL)
# ==============================================================================
# Fase CRISP-DM: Preparación de Datos — Tratamiento de nulos implícitos
#
# PROBLEMA: El distrito Jesenik (código 69) tiene el carácter '?' como valor
# en las columnas A12 (tasa de desempleo 1995) y A15 (número de crímenes 1995).
# Este '?' no es un string de texto: es un nulo implícito del formato original.
#
# SOLUCIÓN: Reemplazamos '?' por NaN y luego imputamos con la mediana del
# resto de los distritos. La mediana es robusta ante la presencia de outliers
# en las variables socioeconómicas.
#
# RELEVANCIA: El salario promedio del distrito, el desempleo y la criminalidad
# son indicadores del entorno económico del cliente, complementarios al
# perfil financiero individual.

def clean_district(df_district):
    """Imputa valores implícitos '?' con la mediana. Renombra columnas."""
    df = df_district.copy()

    # Columnas con '?' confirmados: A12 y A15
    for col in ['A12', 'A15']:
        n_missing = (df[col] == '?').sum()
        df[col] = df[col].replace('?', np.nan).astype(float)
        df[col] = df[col].fillna(df[col].median())
        print(f"   {col}: {n_missing} valores '?' imputados con mediana ({df[col].median():.2f})")

    # Asegurar tipo numérico en el resto
    cols_to_numeric = ['A4','A5','A6','A7','A8','A9','A10','A11','A13','A14','A16']
    for col in cols_to_numeric:
        df[col] = pd.to_numeric(df[col].replace('?', np.nan), errors='coerce')
        df[col] = df[col].fillna(df[col].median())

    # Renombrar columnas codificadas a nombres descriptivos
    rename_map = {
        'A1' : 'district_id',
        'A2' : 'district_name',
        'A3' : 'region',
        'A4' : 'inhabitants',
        'A5' : 'muni_under_499',
        'A6' : 'muni_500_1999',
        'A7' : 'muni_2000_9999',
        'A8' : 'muni_above_10000',
        'A9' : 'cities_count',
        'A10': 'urban_ratio',
        'A11': 'average_salary',
        'A12': 'unemployment_rate_95',
        'A13': 'unemployment_rate_96',
        'A14': 'entrepreneurs_ratio',
        'A15': 'crimes_95',
        'A16': 'crimes_96'
    }
    df = df.rename(columns=rename_map)

    print(f"✅ clean_district: {len(df):,} distritos | Columnas renombradas")
    return df


if district is not None:
    print("Limpiando district...")
    district_clean = clean_district(district)
else:
    district_clean = None
    print("⏭️  district: omitido (USE_DISTRICT=False)")

Limpiando district...
   A12: 1 valores '?' imputados con mediana (2.83)
   A15: 1 valores '?' imputados con mediana (2932.00)
✅ clean_district: 77 distritos | Columnas renombradas


---
## 📊 SECCIÓN 3 — Agregación Transaccional y de Préstamos
**Fase CRISP-DM: 🛠️ Preparación de Datos — Feature Engineering**

Resumimos las tablas de alta cardinalidad (millones de filas) en **1 fila por cuenta**
con indicadores del comportamiento financiero del cliente.

### Features derivados de `transaction` (mismo que v2 + `deposit_ratio`):

| Feature | Qué mide | Relevancia crediticia |
|---------|----------|----------------------|
| `avg_balance` | Saldo promedio | Capacidad de ahorro sostenida |
| `min_balance` | Saldo mínimo | Riesgo de sobregiro |
| `net_flow` | Depósitos − Retiros | Flujo de caja neto |
| `deposit_ratio` | Depósitos / flujo total | Proporción de ingresos sobre total |
| `trans_count` | N° de transacciones | Nivel de actividad financiera |
| `std_balance` | Desv. estándar balance | Estabilidad/volatilidad financiera |

In [8]:
# ==============================================================================
# SECCIÓN 3.1 — AGREGACIÓN DE TRANSACCIONES
# ==============================================================================
# Fase CRISP-DM: Preparación de Datos — Feature Engineering
#
# Comprimimos 1M+ filas de trans en 1 fila por account_id con
# 10 indicadores clave del comportamiento financiero del cliente.

def aggregate_transactions(df_trans):
    """Agrega transacciones por account_id generando features financieros."""
    print("⏳ Agregando transacciones (puede tomar unos segundos)...")
    df = df_trans.copy()

    df['is_deposit']        = df['type'] == 'PRIJEM'
    df['is_withdrawal']     = df['type'].isin(['VYDAJ', 'VYBER'])
    df['deposit_amount']    = np.where(df['is_deposit'],    df['amount'], 0.0)
    df['withdrawal_amount'] = np.where(df['is_withdrawal'], df['amount'], 0.0)

    agg = df.groupby('account_id').agg(
        trans_count       = ('trans_id',          'count'),
        avg_balance       = ('balance',            'mean'),
        min_balance       = ('balance',            'min'),
        max_balance       = ('balance',            'max'),
        std_balance       = ('balance',            'std'),
        avg_trans_amount  = ('amount',             'mean'),
        total_deposits    = ('deposit_amount',     'sum'),
        total_withdrawals = ('withdrawal_amount',  'sum'),
    ).reset_index()

    agg['std_balance'] = agg['std_balance'].fillna(0.0)
    agg['net_flow']    = agg['total_deposits'] - agg['total_withdrawals']

    total_flow           = agg['total_deposits'] + agg['total_withdrawals']
    agg['deposit_ratio'] = np.where(total_flow > 0, agg['total_deposits'] / total_flow, 0.5)

    print(f"✅ trans_agg: {len(agg):,} cuentas × {len(agg.columns)-1} features financieros")
    return agg


trans_agg = aggregate_transactions(trans)

⏳ Agregando transacciones (puede tomar unos segundos)...
✅ trans_agg: 4,500 cuentas × 10 features financieros


In [9]:
# ==============================================================================
# SECCIÓN 3.2 — AGREGACIÓN DE PRÉSTAMOS (OPCIONAL)
# ==============================================================================
# Fase CRISP-DM: Preparación de Datos — Feature Engineering
#
# Historial de endeudamiento: clientes sin préstamo reciben imputación
# de 0 / 'SIN_PRESTAMO' en la fase de merge.
#
# RELEVANCIA: loan_status es el predictor más discriminante del riesgo
# crediticio. Clientes con préstamos al día (A/C) tienen mayor penetración
# de tarjeta que los que incurren en mora (B/D).

def aggregate_loans(df_loan):
    """Agrega préstamos por account_id."""
    loan_agg = df_loan.groupby('account_id').agg(
        loan_count    = ('loan_id',  'count'),
        loan_amount   = ('amount',   'sum'),
        loan_payments = ('payments', 'mean'),
        loan_duration = ('duration', 'max'),
        loan_status   = ('status',   'first')
    ).reset_index()

    print(f"✅ loan_agg: {len(loan_agg):,} cuentas con historial")
    print(f"   Status: {df_loan['status'].value_counts().to_dict()}")
    print("   (A=corriente, B=mora, C=pagado OK, D=pagado con problemas)")
    return loan_agg


if loan is not None:
    loan_agg = aggregate_loans(loan)
else:
    loan_agg = None
    print("⏭️  loan: omitido (USE_LOAN=False)")

✅ loan_agg: 682 cuentas con historial
   Status: {'C': 403, 'A': 203, 'D': 45, 'B': 31}
   (A=corriente, B=mora, C=pagado OK, D=pagado con problemas)


---
## 🔗 SECCIÓN 4 — Integración de Tablas (Merge)
**Fase CRISP-DM: 🛠️ Preparación de Datos — Integración de fuentes**

Construimos la **tabla analítica maestra**: **1 fila = 1 cliente titular (OWNER)**.

### Secuencia de joins en v3:
```
disp[OWNER]  →  client      (INNER: datos demográficos)
             →  card        (LEFT:  has_card + card_type = objetivo)
             →  account     (INNER: características de cuenta)
             →  trans_agg   (LEFT:  features financieros)
             →  loan_agg    (LEFT:  historial crediticio, si USE_LOAN)
             →  district    (LEFT:  contexto socioeconómico, si USE_DISTRICT)
                            ↑
                 join por district_id del client (o del account)
```

In [10]:
district_clean.head(2).T

,0,1
district_id,1,2
district_name,Hl.m. Praha,Benesov
region,Prague,central Bohemia
inhabitants,1204953,88884
muni_under_499,0,80
muni_500_1999,0,26
muni_2000_9999,0,6
muni_above_10000,1,2
cities_count,1,5
urban_ratio,100.0,46.7


In [11]:
# ==============================================================================
# SECCIÓN 4 — INTEGRACIÓN DE TABLAS
# ==============================================================================
# Fase CRISP-DM: Preparación de Datos — Integración de fuentes

def merge_tables(client_c, disp, account_c, trans_agg,
                 card, loan_agg=None, district_c=None):
    """Integra todas las tablas en una tabla analítica maestra."""
    print("🔗 Integrando tablas...")

    # ── PASO 1: Filtrar solo titulares OWNER ───────────────────────────────────
    # Solo titulares toman decisiones financieras y son sujetos de análisis.
    # Garantiza: 1 disp_id = 1 cliente = 1 cuenta.
    owners = disp[disp['type'] == 'OWNER'][['disp_id', 'client_id', 'account_id']].copy()
    print(f"   OWNER filtrados           : {len(owners):,}")

    # ── PASO 2: Agregar datos demográficos del cliente ─────────────────────────
    df = owners.merge(client_c, on='client_id', how='inner')
    print(f"   Tras join con client      : {len(df):,} filas")

    # ── PASO 3: Variable objetivo → has_card ──────────────────────────────────
    # has_card = 1 → cliente con tarjeta (positivo del modelo)
    # has_card = 0 → cliente sin tarjeta (negativo del modelo)
    df = df.merge(card[['disp_id', 'card_id', 'type']], on='disp_id', how='left')
    df['has_card']  = np.where(df['card_id'].notna(), 1, 0)   # ← VARIABLE OBJETIVO
    df['card_type'] = df['type'].fillna('sin_tarjeta')
    df = df.drop(columns=['card_id', 'type'])
    print(f"   has_card=1 (con tarjeta)  : {df['has_card'].sum():,}")
    print(f"   has_card=0 (sin tarjeta)  : {(df['has_card']==0).sum():,}")

    # ── PASO 4: Características de la cuenta ──────────────────────────────────
    df = df.merge(account_c, on='account_id', how='inner')
    print(f"   Tras join con account     : {len(df):,} filas")

    # ── PASO 5: Features de comportamiento transaccional ──────────────────────
    df = df.merge(trans_agg, on='account_id', how='left')
    trans_cols = ['trans_count','avg_balance','min_balance','max_balance',
                  'std_balance','avg_trans_amount','total_deposits',
                  'total_withdrawals','net_flow','deposit_ratio']
    for col in trans_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0.0)
    print(f"   Tras join con trans_agg   : {len(df):,} filas")

    # ── PASO 6: Historial de préstamos (OPCIONAL) ──────────────────────────────
    if loan_agg is not None:
        df = df.merge(loan_agg, on='account_id', how='left')
        df['has_loan']     = np.where(df['loan_count'].notna(), 1, 0)
        df['loan_amount']  = df['loan_amount'].fillna(0.0)
        df['loan_payments']= df['loan_payments'].fillna(0.0)
        df['loan_duration']= df['loan_duration'].fillna(0).astype(int)
        df['loan_status']  = df['loan_status'].fillna('SIN_PRESTAMO')
        df = df.drop(columns=['loan_count'], errors='ignore')
        print(f"   Tras join con loan_agg    : {len(df):,} filas")
    else:
        print("   ⏭️  loan_agg: omitido")

    # ── PASO 7: Contexto socioeconómico del distrito (OPCIONAL) ───────────────
    # El cliente está asociado a un distrito a través de su 'district_id'.
    # Las variables del distrito capturan el entorno macroeconómico local:
    # salario promedio, tasas de desempleo y criminalidad, región geográfica.
    df.head(1)
    if district_c is not None:
        # district_id viene del cliente (donde nació) o de la cuenta (donde abrió)
        # En Berka, client.district_id indica el distrito del cliente
        df = df.merge(
            district_c[['district_id', 'region', 'inhabitants', 'average_salary',
                         'unemployment_rate_95', 'unemployment_rate_96',
                         'entrepreneurs_ratio', 'crimes_95', 'crimes_96',
                         'urban_ratio', 'cities_count']],
            left_on='district_id_x', right_on='district_id', how='left'
        )
        # Imputar con mediana si algún district_id no tiene match
        dist_num_cols = ['inhabitants','average_salary','unemployment_rate_95',
                         'unemployment_rate_96','entrepreneurs_ratio',
                         'crimes_95','crimes_96','urban_ratio','cities_count']
        for col in dist_num_cols:
            if col in df.columns:
                df[col] = df[col].fillna(df[col].median())
        if 'region' in df.columns:
            df['region'] = df['region'].fillna('Desconocido')
        print(f"   Tras join con district    : {len(df):,} filas")
        print(f"   Regiones encontradas      : {df['region'].nunique()}")
    else:
        print("   ⏭️  district: omitido")

    # ── PASO 8: Eliminar columnas técnicas ─────────────────────────────────────
    drop_cols = ['disp_id', 'client_id', 'account_id', 'district_id']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    print(f"\n✅ Tabla consolidada: {df.shape[0]:,} clientes × {df.shape[1]} variables")
    return df


df_merged = merge_tables(
    client_clean, disp, account_clean, trans_agg,
    card, loan_agg, district_clean
)

🔗 Integrando tablas...
   OWNER filtrados           : 4,500
   Tras join con client      : 4,500 filas
   has_card=1 (con tarjeta)  : 892
   has_card=0 (sin tarjeta)  : 3,608
   Tras join con account     : 4,500 filas
   Tras join con trans_agg   : 4,500 filas
   Tras join con loan_agg    : 4,500 filas
   Tras join con district    : 4,500 filas
   Regiones encontradas      : 8

✅ Tabla consolidada: 4,500 clientes × 35 variables


In [12]:
df_merged["district_equal"] = df_merged["district_id_x"] == df_merged["district_id_y"]

In [13]:
df_merged[df_merged["district_id_x"] == df_merged["district_id_y"]]

,district_id_x,gender,birth_month,birth_year,age,has_card,card_type,district_id_y,frequency,account_age_months,...,inhabitants,average_salary,unemployment_rate_95,unemployment_rate_96,entrepreneurs_ratio,crimes_95,crimes_96,urban_ratio,cities_count,district_equal
0,18,F,12,1970,28,0,sin_tarjeta,18,Mensual,45,...,70699,8968,2.83,3.35,131,1740.0,1910,65.3,4,True
1,1,M,2,1945,53,0,sin_tarjeta,1,Mensual,70,...,1204953,12541,0.29,0.43,167,85677.0,99107,100.0,1,True
2,5,M,12,1956,42,0,sin_tarjeta,5,Mensual,17,...,95616,9307,3.85,4.43,118,2616.0,3040,51.4,6,True
3,12,M,9,1919,79,0,sin_tarjeta,12,Mensual,34,...,107870,8754,3.83,4.31,137,3804.0,3868,58.0,6,True
4,15,M,1,1929,69,0,sin_tarjeta,15,Mensual,19,...,58796,9045,3.13,3.60,124,1845.0,1879,51.9,5,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4495,8,M,1,1942,56,0,sin_tarjeta,8,Mensual,55,...,112065,11277,1.25,1.44,127,5179.0,4987,69.4,8,True
4496,1,F,10,1945,53,0,sin_tarjeta,1,Semanal,43,...,1204953,12541,0.29,0.43,167,85677.0,99107,100.0,1,True
4497,61,M,4,1968,30,1,classic,61,Mensual,50,...,117897,8814,4.76,5.74,107,2112.0,2059,53.8,6,True
4498,67,F,10,1962,36,0,sin_tarjeta,67,Mensual,38,...,106054,8110,5.77,6.55,109,3244.0,3079,63.1,6,True


In [14]:
df_merged["district_id_x"].unique()

array([18,  1,  5, 12, 15, 51, 60, 57, 40, 54, 76, 21, 47, 46, 43, 74, 30,
       68, 52, 73,  7, 36, 22, 75, 38, 20, 19, 16, 72, 39, 32, 64, 45, 67,
       77,  4, 50, 29, 70, 37,  8, 63,  3, 26, 33, 44, 31, 17,  2, 55, 41,
       35, 11, 53,  6, 27, 65, 13, 34, 24, 10, 58, 28, 59, 66, 71, 48, 69,
       61, 56, 25, 49, 14, 62, 42, 23,  9])

In [15]:
df_merged["district_id_y"].unique()

array([18,  1,  5, 12, 15, 51, 60, 57, 70, 54, 76, 21, 47, 43, 74, 30, 68,
       52, 73,  7, 36, 22, 67, 75, 20, 19, 64, 72, 39, 32, 45, 16, 77,  4,
       50, 29, 48, 37,  8, 63,  3, 26, 44, 31, 17,  2, 55, 41, 35, 11, 53,
        6, 27, 65, 40, 34, 38, 24, 10, 46, 58, 23, 28, 25, 59, 13, 66, 71,
       69, 61, 56, 49, 14, 62, 42,  9, 33])

In [16]:
len(df_merged["district_id_x"].unique())

77

In [17]:
df_merged.head(10).T

,0,1,2,3,4,5,6,7,8,9
district_id_x,18,1,5,12,15,51,60,57,40,54
gender,F,M,M,M,M,F,M,M,M,F
birth_month,12,2,12,9,1,2,10,5,2,5
birth_year,1970,1945,1956,1919,1929,1938,1935,1943,1981,1974
age,28,53,42,79,69,60,63,55,17,24
has_card,0,0,0,0,0,0,1,0,0,0
card_type,sin_tarjeta,sin_tarjeta,sin_tarjeta,sin_tarjeta,sin_tarjeta,sin_tarjeta,gold,sin_tarjeta,sin_tarjeta,sin_tarjeta
district_id_y,18,1,5,12,15,51,60,57,70,54
frequency,Mensual,Mensual,Mensual,Mensual,Mensual,Mensual,Mensual,Mensual,Mensual,Mensual
account_age_months,45,70,17,34,19,51,25,39,71,28


In [18]:
df_merged.head(2).T

,0,1
district_id_x,18,1
gender,F,M
birth_month,12,2
birth_year,1970,1945
age,28,53
has_card,0,0
card_type,sin_tarjeta,sin_tarjeta
district_id_y,18,1
frequency,Mensual,Mensual
account_age_months,45,70


In [19]:
# ==============================================================================
# SECCIÓN 4.1 — REVISIÓN POST-MERGE
# ==============================================================================
# Fase CRISP-DM: Validación de la tabla consolidada

print("Dimensiones:", df_merged.shape)
print("\nNulos por columna (solo >0):")
nulos = df_merged.isnull().sum()
nulos_reales = nulos[nulos > 0]
print("   ✅ Sin nulos" if len(nulos_reales) == 0 else nulos_reales.to_string())

print("\nColumnas disponibles:")
for i, col in enumerate(df_merged.columns, 1):
    print(f"   {i:2d}. {col}  [{df_merged[col].dtype}]")

Dimensiones: (4500, 36)

Nulos por columna (solo >0):
   ✅ Sin nulos

Columnas disponibles:
    1. district_id_x  [int64]
    2. gender  [object]
    3. birth_month  [int64]
    4. birth_year  [int64]
    5. age  [int64]
    6. has_card  [int64]
    7. card_type  [object]
    8. district_id_y  [int64]
    9. frequency  [object]
   10. account_age_months  [int64]
   11. trans_count  [int64]
   12. avg_balance  [float64]
   13. min_balance  [float64]
   14. max_balance  [float64]
   15. std_balance  [float64]
   16. avg_trans_amount  [float64]
   17. total_deposits  [float64]
   18. total_withdrawals  [float64]
   19. net_flow  [float64]
   20. deposit_ratio  [float64]
   21. loan_amount  [float64]
   22. loan_payments  [float64]
   23. loan_duration  [int64]
   24. loan_status  [object]
   25. has_loan  [int64]
   26. region  [object]
   27. inhabitants  [int64]
   28. average_salary  [int64]
   29. unemployment_rate_95  [float64]
   30. unemployment_rate_96  [float64]
   31. entreprene

---
## ⚙️ SECCIÓN 5 — Preparación Final para Modelado
**Fase CRISP-DM: 🛠️ Preparación de Datos — Transformaciones para ML**

4 pasos secuenciales para compatibilizar los datos con cualquier algoritmo de ML:

```
Tabla consolidada
    │
    ├─ [5.1] Nulos residuales → eliminar cols >60% NaN
    │
    ├─ [5.2] Outliers → Winsorización ±3σ
    │         (sin perder registros, estabiliza distribuciones financieras)
    │
    ├─ [5.3] Estandarización → Z-Score (Z = (X−μ)/σ → media=0, σ=1)
    │         (necesario para SVM, KNN, Redes Neuronales)
    │
    └─ [5.4] Categóricas → One-Hot Encoding (0/1)
               (gender, frequency, loan_status, card_type, region)
               │
               └─▶ TABLA MINABLE FINAL v3
```

In [20]:
# ==============================================================================
# SECCIÓN 5 — PREPARACIÓN FINAL PARA MODELADO
# ==============================================================================
# Fase CRISP-DM: Preparación de Datos — Transformaciones para ML

def prepare_data(df, use_district=True):
    """Aplica: Nulos → Winsorización → Z-Score → One-Hot Encoding."""
    print("⚙️  Iniciando preparación final de datos v3...\n")
    df = df.copy()

    # ──────────────────────────────────────────────────────────────────────────
    # PASO 5.1 — TRATAMIENTO DE NULOS
    # ──────────────────────────────────────────────────────────────────────────
    print("[5.1] Tratamiento de nulos")
    null_pct = df.isnull().mean() * 100
    cols_drop = null_pct[null_pct > 60].index.tolist()
    if cols_drop:
        print(f"      ⚠️  Eliminando {len(cols_drop)} cols con >60% nulos: {cols_drop}")
        df = df.drop(columns=cols_drop)
    else:
        print("      ✅ Ninguna columna supera el 60% de nulos.")

    # Imputar nulos numéricos residuales con mediana
    num_nulos = df.select_dtypes(include='number').columns[df.select_dtypes(include='number').isnull().any()]
    for col in num_nulos:
        df[col] = df[col].fillna(df[col].median())
    if len(num_nulos) > 0:
        print(f"      📝 Imputados con mediana: {list(num_nulos)}")

    df_eda = df.copy()  # Copia pre-estandarización para EDA

    # ──────────────────────────────────────────────────────────────────────────
    # PASO 5.2 — OUTLIERS: WINSORIZACIÓN a ±3σ
    # ──────────────────────────────────────────────────────────────────────────
    # Recortamos extremos en variables financieras y socioeconómicas.
    # No se eliminan registros: solo se truncan los valores al límite ±3σ.
    print("\n[5.2] Outliers — Winsorización ±3σ")

    # Variables del perfil individual
    numeric_base = [
        'age', 'birth_month', 'account_age_months',
        'trans_count', 'avg_balance', 'min_balance', 'max_balance',
        'std_balance', 'avg_trans_amount', 'total_deposits',
        'total_withdrawals', 'net_flow', 'deposit_ratio',
        'loan_amount', 'loan_payments', 'loan_duration'
    ]

    # Variables del distrito (solo si se usa)
    numeric_district = [
        'inhabitants', 'average_salary', 'unemployment_rate_95',
        'unemployment_rate_96', 'entrepreneurs_ratio',
        'crimes_95', 'crimes_96', 'urban_ratio', 'cities_count'
    ] if use_district else []

    numeric_cols = [c for c in numeric_base + numeric_district if c in df.columns]

    total_recortados = 0
    for col in numeric_cols:
        mu, sigma = df[col].mean(), df[col].std()
        if sigma > 0:
            lo, hi = mu - 3*sigma, mu + 3*sigma
            n_out  = ((df[col] < lo) | (df[col] > hi)).sum()
            total_recortados += n_out
            df[col] = np.clip(df[col], lo, hi)

    print(f"      ✅ Winsorización en {len(numeric_cols)} variables.")
    print(f"      📊 Valores recortados: {total_recortados:,}")

    # ──────────────────────────────────────────────────────────────────────────
    # PASO 5.3 — ESTANDARIZACIÓN: Z-SCORE
    # ──────────────────────────────────────────────────────────────────────────
    # Z = (X - μ) / σ → media=0, σ=1 en todas las numéricas.
    # Prefijo 'z_' en las columnas resultantes.
    # NECESARIO para: SVM, KNN, Redes Neuronales, Regresión Logística.
    print("\n[5.3] Z-Score (Z = (X − μ) / σ)")

    for col in numeric_cols:
        mu, sigma = df[col].mean(), df[col].std()
        df[f'z_{col}'] = (df[col] - mu) / (sigma if sigma > 0 else 1.0)

    print(f"      ✅ {len(numeric_cols)} variables estandarizadas (columnas 'z_*').")

    # ──────────────────────────────────────────────────────────────────────────
    # PASO 5.4 — ONE-HOT ENCODING
    # ──────────────────────────────────────────────────────────────────────────
    # Convierte categóricas en columnas binarias (0/1).
    # drop_first=True elimina la primera categoría para evitar multicolinealidad.
    #
    # v3 agrega 'region' (proveniente de district) a las categóricas.
    print("\n[5.4] One-Hot Encoding para variables categóricas")

    categorical_base    = ['gender', 'frequency', 'loan_status', 'card_type']
    categorical_district = ['region'] if use_district else []
    categorical_cols    = [c for c in categorical_base + categorical_district if c in df.columns]

    print(f"      Variables a codificar: {categorical_cols}")
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
    print(f"      ✅ One-Hot aplicado. Dimensiones finales: {df.shape}")

    return df, df_eda


df_final, df_eda = prepare_data(df_merged, use_district=USE_DISTRICT)

⚙️  Iniciando preparación final de datos v3...

[5.1] Tratamiento de nulos
      ✅ Ninguna columna supera el 60% de nulos.

[5.2] Outliers — Winsorización ±3σ
      ✅ Winsorización en 25 variables.
      📊 Valores recortados: 860

[5.3] Z-Score (Z = (X − μ) / σ)
      ✅ 25 variables estandarizadas (columnas 'z_*').

[5.4] One-Hot Encoding para variables categóricas
      Variables a codificar: ['gender', 'frequency', 'loan_status', 'card_type', 'region']
      ✅ One-Hot aplicado. Dimensiones finales: (4500, 73)


---
## 💾 SECCIÓN 6 — Exportación y Resumen Final
**Fase CRISP-DM: 🛠️ Preparación de Datos (cierre) → Pre-Modelado**

Exportamos la tabla minable v3. Esta es la entrada directa para la **Fase de Modelado**.

### Próximos pasos en CRISP-DM (Fase de Modelado):
1. `train_test_split` → 80% entrenamiento / 20% prueba (estratificado por `has_card`)
2. Entrenar: `LogisticRegression`, `RandomForestClassifier`, `XGBClassifier`, MLP (Keras)
3. Evaluar: `AUC-ROC`, `F1`, `Precision`, `Recall`, `Confusion Matrix`
4. Si desbalance severo: aplicar `class_weight='balanced'` o `SMOTE`

In [21]:
# ==============================================================================
# SECCIÓN 6 — EXPORTACIÓN DE LA TABLA MINABLE v3
# ==============================================================================
# Fase CRISP-DM: Pre-Modelado

df_final.to_csv(OUTPUT_CSV, index=False, sep=";")

print("=" * 65)
print("✅ ¡PIPELINE v3 COMPLETADO EXITOSAMENTE!")
print("=" * 65)
print(f"📁 Archivo exportado    : {OUTPUT_CSV}")
print(f"📐 Dimensiones          : {df_final.shape[0]:,} clientes × {df_final.shape[1]} variables")
print()
print("── Variable objetivo (has_card) ──")
vc = df_final['has_card'].value_counts()
pct = df_final['has_card'].value_counts(normalize=True) * 100
print(f"   has_card = 1 : {vc[1]:,}  ({pct[1]:.1f}%)")
print(f"   has_card = 0 : {vc[0]:,} ({pct[0]:.1f}%)")
print()
print("── Resumen de variables ──")
z_cols   = [c for c in df_final.columns if c.startswith('z_')]
ohe_cols = [c for c in df_final.columns if c not in z_cols
            and df_final[c].nunique() == 2
            and c not in ['has_card', 'has_loan']]
num_orig = [c for c in df_final.columns if c not in z_cols
            and df_final[c].dtype in ['float64','int64']
            and c not in ohe_cols]
print(f"   Z-Score (z_*)           : {len(z_cols)}")
print(f"   One-Hot encoding         : {len(ohe_cols)}")
print(f"   Numéricas originales     : {len(num_orig)}")
print()
print("── Tablas utilizadas ──")
print(f"   client + disp + account + trans + card : ✅ obligatorias")
print(f"   loan    : {'✅ incluida' if USE_LOAN else '⏭️  omitida'}")
print(f"   district: {'✅ incluida' if USE_DISTRICT else '⏭️  omitida'}")
print()
print("── Siguiente paso CRISP-DM: MODELADO ──")
print("   from sklearn.model_selection import train_test_split")
print("   X = df_final.drop('has_card', axis=1)")
print("   y = df_final['has_card']")
print("   X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)")

✅ ¡PIPELINE v3 COMPLETADO EXITOSAMENTE!
📁 Archivo exportado    : minable_banco_berka_v3.csv
📐 Dimensiones          : 4,500 clientes × 73 variables

── Variable objetivo (has_card) ──
   has_card = 1 : 892  (19.8%)
   has_card = 0 : 3,608 (80.2%)

── Resumen de variables ──
   Z-Score (z_*)           : 25
   One-Hot encoding         : 18
   Numéricas originales     : 30

── Tablas utilizadas ──
   client + disp + account + trans + card : ✅ obligatorias
   loan    : ✅ incluida
   district: ✅ incluida

── Siguiente paso CRISP-DM: MODELADO ──
   from sklearn.model_selection import train_test_split
   X = df_final.drop('has_card', axis=1)
   y = df_final['has_card']
   X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)


---
## 📊 Resumen del Pipeline v3

| Sección | Fase CRISP-DM | Tablas involucradas | Resultado |
|---------|--------------|--------------------|-----------|
| **0** | Preparación técnica | — | Entorno + flags opcionales |
| **1** | 🔍 Comprensión de Datos | 5 oblig. + 2 opcionales | 7 tablas cargadas, balance de clases analizado |
| **2** | 🛠️ Prep. — Limpieza | `client`, `account`, `district` | `gender`, `age`, `account_age_months`; nulos `?` imputados |
| **3** | 🛠️ Prep. — Feature Eng. | `trans`, `loan` | 10 features financieros + historial crediticio |
| **4** | 🛠️ Prep. — Integración | Todas | 1 fila = 1 cliente; `has_card` como objetivo |
| **5** | 🛠️ Prep. — para ML | Tabla consolidada | Nulos ✓ / Winsorización ✓ / Z-Score ✓ / One-Hot ✓ |
| **6** | 🚀 Pre-Modelado | — | `minable_banco_berka_v3.csv` exportado |

---

### Comparativa de versiones

| Característica | v1 | v2 | v3 |
|---------------|:--:|:--:|:--:|
| Tablas obligatorias | 7 | 5 | 5 |
| `district` | ✅ | ❌ | ⚙️ Opcional |
| `loan` | ✅ | ⚙️ Opcional | ⚙️ Opcional |
| `deposit_ratio` | ❌ | ✅ | ✅ |
| `card_type` | ❌ | ✅ | ✅ |
| Prefijo Z-Score | `std_` | `z_` | `z_` |
| `region` en One-Hot | ✅ | ❌ | ⚙️ Si district activo |
| Flags de control | ❌ | ❌ | ✅ |

---
*Caso de uso: **"Predecir qué clientes tienen alta probabilidad de ser buenos candidatos para una tarjeta de crédito."***